# S6E6 — Colab runner (idempotent)

A **thin, stable launcher.** It only mounts Drive, sets secrets, and **fresh-clones** the
repo — then runs `colab/bootstrap.py`, which holds ALL the run logic (install, data, vendor,
run, dashboard, Drive-sync, diary push) and is pulled fresh every time. **You never edit this
notebook, and you never re-upload anything.**

- **Run an experiment:** Runtime → Run all (cells 1–2).
- **Switch experiment:** edit `SPRINT_ACTIVE.txt` in the repo, push, re-run cell 2.
- **Submit:** cell 4 (puts a version on the leaderboard). **Download:** cell 5 (optional).
- **Secrets (🔑, notebook access ON):** `KAGGLE_API_TOKEN` (required), `GH_TOKEN` (optional, diary push).

## 1. Setup — Drive + secrets (stable; rarely changes)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/Colab Notebooks/kaggle/s6e6'
os.makedirs(os.environ['DRIVE_ROOT'], exist_ok=True)

from google.colab import userdata
for name in ('KAGGLE_API_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(name)
        if v:
            os.environ[name] = v; print(f'✓ {name} set')
        else:
            print(f'⚠ {name} empty')
    except Exception:
        req = ' (REQUIRED)' if name == 'KAGGLE_API_TOKEN' else ' (optional)'
        print(f'⚠ no {name}{req} — add via 🔑, notebook access ON')


## 2. Run — fresh clone + bootstrap (all logic lives in the repo)

In [ ]:
%cd /content
!rm -rf playground-s6e6
!git clone -q https://github.com/SirGrigor/playground-s6e6.git
%cd playground-s6e6
!git log -1 --oneline
!python colab/bootstrap.py

## 3. Review — open the HTML dashboard inline

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
p = Path('/content/playground-s6e6/reports/dashboard.html')
display(HTML(p.read_text())) if p.exists() else print('no dashboard.html yet')

## 4. Submit to the leaderboard  ←  RUN THIS to submit

The Kaggle API is already authenticated in Colab. Set `SUBMIT_FILE` to the version to put up
(run cell 2 first so the file is present). Daily limit ~5/day. Prints your public score —
record it against the holdout to track CV↔LB.

In [ ]:
# submissions/ holds: v5_decision.csv, v6_ensemble.csv, ... (written this run + pulled from Drive)
SUBMIT_FILE = 'v5_decision.csv'
SUBMIT_MSG  = 'v5 decision-rule: natural probs + bal-acc threshold tuning (holdout 0.9657)'
import os
p = f'/content/playground-s6e6/submissions/{SUBMIT_FILE}'
if not os.path.exists(p):
    print(f'⚠ {p} not found — run cell 2 first (it pulls submissions/ from Drive), or check the name.')
else:
    !cd /content/playground-s6e6 && kaggle competitions submit -c playground-series-s6e6 -f submissions/{SUBMIT_FILE} -m "{SUBMIT_MSG}"
    import time; time.sleep(10)
    print('\n--- recent submissions (public score) ---')
    !kaggle competitions submissions -c playground-series-s6e6 | head -10

## 5. (optional) Download the newest submission to your computer

In [ ]:
from pathlib import Path
from google.colab import files
subs = sorted(Path('/content/playground-s6e6/submissions').glob('*.csv'), key=lambda p: -p.stat().st_mtime)
if subs:
    print('newest:', subs[0].name); files.download(str(subs[0]))
else:
    print('no submissions yet')